# Algoritmos de Regresion - Prediccion de Precio
**Camila Rojas**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Semana12_Regresion_Camila_Rojas").getOrCreate()

# Cargar datos generados en Semana 10 (Clustering)
df_clusters = spark.read.parquet("/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans")

print(f"Datos cargados: {df_clusters.count()}")

# Eliminar columna prediction del clustering anterior
df_regresion = df_clusters.drop("prediction")

print("Columnas disponibles para regresion:")
print(df_regresion.columns)

Datos cargados: 4415
Columnas disponibles para regresion:
['precio_noche', 'estrellas', 'puntuacion', 'noches', 'features', 'scaledFeatures']


In [2]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Variables predictoras (excluir precio_noche que es la variable objetivo)
variables_regresion = ["estrellas", "puntuacion", "ciudad_cat", "tipo_alojamiento_cat"]

# Verificar que todas las variables existen
for var in variables_regresion:
    if var not in df_regresion.columns:
        print(f"ADVERTENCIA: {var} no existe en los datos")

# Preparar columnas numericas
df_num = df_regresion.select(
    col("precio_noche").cast("double").alias("label_precio"),
    col("estrellas").cast("double").alias("estrellas"),
    col("puntuacion").cast("double").alias("puntuacion"),
    col("ciudad_cat").cast("double").alias("ciudad_cat"),
    col("tipo_alojamiento_cat").cast("double").alias("tipo_alojamiento_cat")
).na.fill(0)

variables_regresion = ["estrellas", "puntuacion", "ciudad_cat", "tipo_alojamiento_cat"]

# Vectorizar
assembler = VectorAssembler(inputCols=variables_regresion, outputCol="features_reg")
df_vector = assembler.transform(df_num)

# Escalar
scaler = StandardScaler(inputCol="features_reg", outputCol="scaledFeatures_reg", withStd=True, withMean=False)
scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

# Dividir en entrenamiento y prueba
train_reg, test_reg = df_scaled.randomSplit([0.7, 0.3], seed=42)

print(f"Entrenamiento: {train_reg.count()}")
print(f"Prueba: {test_reg.count()}")

ADVERTENCIA: ciudad_cat no existe en los datos
ADVERTENCIA: tipo_alojamiento_cat no existe en los datos


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ciudad_cat` cannot be resolved. Did you mean one of the following? [`estrellas`, `features`, `noches`, `puntuacion`, `precio_noche`].;
'Project [cast(precio_noche#0 as double) AS label_precio#33, cast(estrellas#1 as double) AS estrellas#34, cast(puntuacion#2 as double) AS puntuacion#35, cast('ciudad_cat as double) AS ciudad_cat#36, cast('tipo_alojamiento_cat as double) AS tipo_alojamiento_cat#37]
+- Project [precio_noche#0, estrellas#1, puntuacion#2, noches#3, features#4, scaledFeatures#5]
   +- Relation [precio_noche#0,estrellas#1,puntuacion#2,noches#3,features#4,scaledFeatures#5,prediction#6] parquet


In [ ]:
# Regresion Lineal - Prediccion de Precio por Noche
lr_reg = LinearRegression(featuresCol="scaledFeatures_reg", labelCol="label_precio", maxIter=100)
lr_model = lr_reg.fit(train_reg)
predictions = lr_model.transform(test_reg)

# Evaluar
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)

print("=" * 50)
print("REGRESION LINEAL - PREDICCION DE PRECIO")
print("=" * 50)
print(f"R2:  {r2 * 100:.2f}%")
print(f"RMSE: ${rmse:,.0f} CLP")
print("=" * 50)

In [ ]:
# Coeficientes del modelo
print(f"Intercepto (precio base): ${lr_model.intercept:,.0f} CLP")
print("Coeficientes por variable:")
for i, var in enumerate(variables_regresion):
    print(f"  {var}: ${lr_model.coefficients[i]:,.0f} CLP")

In [ ]:
# Comparativa: precio real vs precio predicho
predictions.select("label_precio", "prediction").show(10)